# 多人会议语音智能解析系统
## 一、绪论
### 1.1 项目背景与意义
随着线上会议、远程办公的普及，海量会议语音数据需要人工整理，效率低下且易出错。传统语音系统仅实现语音转写，无法区分不同发言者、识别发言情绪与说话人属性。

针对该痛点，本项目设计端到端多人会议语音智能解析系统，融合声纹识别、音频多分类、语音转写技术，自动完成说话人区分、语音转文字、情绪 / 性别 / 年龄识别，并生成结构化会议日记。同时针对情绪变化干扰声纹特征的行业痛点，设计情感偏移补偿策略，提升复杂情绪场景下说话人识别的鲁棒性。
### 1.2 项目功能定位
结合需求与开发内容，系统核心功能分为四大模块：
- 说话人管理：支持手动注册永久说话人、自动注册临时说话人、临时说话人转正、说话人信息编辑 / 删除；
- 语音解析：支持本地音频文件解析、实时麦克风录音解析，输出时间戳 + 说话人 + 情绪 + 性别 + 年龄 + 文本结构化信息；
- 算法增强：基于 CREMA-D 数据集统计情感偏移量，对声纹特征做补偿，降低情绪对识别结果的干扰；
- 交互与导出：可视化前端界面、对话内容在线编辑、会议日记 TXT 文件导出
### 1.3 整体技术路线
整体技术栈分为数据层、算法模型层、业务逻辑层、前后端交互层：
- 数据层：TIMIT（说话人训练）、CREMA-D（情感 / 性别 / 年龄训练）；
- 特征层：FBank 梅尔频谱特征、原始语音波形；
- 模型层：ECAPA-TDNN（说话人识别）、Wav2Vec 2.0（多任务音频分类）、Whisper（语音转写）；
- 后端：Python + PyTorch + Flask；
- 前端：HTML + CSS + JavaScript + MediaRecorder（录音）；
- 优化策略：情感偏移补偿、VAD 语音活动检测、队列异步处理、声纹移动平均更新
### 1.4 报告组织结构
本文共分为九大章节：绪论、系统总体架构、实验环境与数据集、核心模块原理与代码解析、模型训练过程、系统功能流程、实验结果与分析、问题排查与优化、总结与展望

## 二、系统总体架构与数据流
### 2.1 整体框架图
```text
+---------------------------------------------+
|                前端交互层                    |
|  音频上传 / 实时录音 / 说话人管理 / 结果展示  |
|  HTML + CSS + JavaScript + MediaRecorder    |
+----------------------+----------------------+
                       | HTTP请求/文件流
+----------------------v----------------------+
|                Flask 后端服务层              |
|  路由接口 / 音频格式转换 / 任务调度 / 队列管理 |
+----------------------+----------------------+
                       | 分发给各算法模块
+----------------------v----------------------+
|                核心算法层                    |
|  ┌─────────┐  ┌──────────┐  ┌────────────┐  |
|  │Whisper  │  │ECAPA-TDNN│  │Wav2Vec2.0 │  |
|  │语音转写 │  │说话人识别│  │情感/性别/ │  |
|  │(ASR)    │  │+情感补偿 │  │年龄识别   │  |
|  └─────────┘  └──────────┘  └────────────┘  |
+----------------------+----------------------+
                       | 特征/标签交互
+----------------------v----------------------+
|                数据与存储层                  |
|  数据集 / 模型权重 / 说话人声纹库 / 情感偏移量 |
+---------------------------------------------+
```
![整体框架图](record/user_use.png)

### 2.2 核心业务数据流
系统分为本地音频解析和实时录音解析两大使用场景，数据流略有差异
#### 2.2.1 本地音频解析数据流
1. 前端选择音频文件 → 表单上传至后端 /api/recognize
2. 后端统一转换为标准WAV格式音频
3. Whisper ASR 分割音频，输出带时间戳的文本片段
4. 逐片段处理：
   - 截取对应时间段音频 → 提取FBank特征
   - ECAPA-TDNN 提取声纹Embedding
   - Wav2Vec2 识别当前片段情感、性别、年龄
   - 加载情感偏移量 → 对声纹做补偿校正
   - 声纹与数据库做余弦相似度匹配 → 判定说话人（永久/临时/未知）
5. 合并相邻同说话人片段 → 结构化结果回传前端
6. 前端渲染对话列表、说话人列表，支持编辑与导出

#### 2.2.2 实时录音解析数据流
1. 前端请求麦克风权限 → 启动VAD音量检测（100ms/次，静音阈值600ms）
2. 检测到人声 → 新建MediaRecorder录制片段
3. 检测到超长静音 → 结束当前片段，分配序号，异步上传后端
4. 后端接收音频片段 → 执行与本地解析一致的算法流程
5. 前端维护解析队列+等待列表，严格按片段序号顺序展示结果（解决异步乱序问题）
6. 循环执行人声检测-录制-上传-解析，直至手动停止录音
7. 录音结束 → 清空资源，汇总全部会议内容，生成日记

### 2.3 模块依赖关系
1. 基础预处理模块：extract_fbank.py 为 ECAPA-TDNN 提供特征输入；
2. 数据集模块：timit_dataset.py/wav2vec2_dataset.py 分别服务两大模型训练；
3. 核心模型模块：三大模型相互独立，通过main.py完成集成调度；
4. 补偿模块：compute_emotion_bias.py 预计算情感偏移量，为emotion_compensated_reco.py提供补偿依据；
5. 接口层：app.py封装所有功能接口，对接前端


## 三、实验环境与数据集
### 3.1 软硬件环境
#### 3.1.1 硬件环境
- 计算设备：NVIDIA GPU（CUDA 可用）/ 普通 CPU
- 音频设备：麦克风（实时录音使用）
- 存储：模型权重、数据集、声纹数据库本地存储
#### 3.1.2 软件环境
- 操作系统：Windows 10 / Windows 11
- 编程语言：Python 3.8+
- 核心依赖库：
  - 深度学习框架：torch、torchaudio
  - 音频处理：librosa、pydub、wave
  - 大模型：transformers（Wav2Vec2）、openai-whisper
  - 后端服务：Flask
  - 数据处理：pandas、numpy、json
  - 前端：原生 HTML/CSS/JavaScript
#### 3.1.3 全局配置
系统统一音频与模型超参数，全局配置如下：
```python
n_mels=40        # FBank梅尔滤波器数量
max_len=500      # 音频特征最大帧长
sr=16000         # 统一采样率16kHz
batch_size=32    # 训练批次大小
epochs=50        # ECAPA-TDNN训练轮数
lr=0.001         # 初始学习率
device=CUDA/CPU  # 自动选择计算设备
```

### 3.2 数据集介绍
本项目使用两个标准公开语音数据集
#### 3.2.1 TIMIT 数据集（说话人识别专用）
1. 用途：训练、验证、测试 ECAPA-TDNN 说话人识别模型；
2. 基本信息：美式英语语音数据集，包含462 名不同说话人，音频采样率 16kHz，环境纯净、信噪比高；
3. 预处理文件：preprocess_timit.py 解析路径、说话人 ID、性别，生成open_test_info.json索引文件；
4. 数据集加载类：timit_dataset.py 读取索引、提取 FBank 特征、生成标签映射。
#### 3.2.2 CREMA-D 数据集（多任务分类专用）
1. 用途：训练 Wav2Vec2 情感 / 性别 / 年龄多任务模型，同时用于计算情感偏移量；
2. 基本信息：包含 6 类情感、2 种性别、3 个年龄段，每条语音带有明确情感标签；
3. 标签定义：
   - 情感：ANG (愤怒)、DIS (厌恶)、FEA (恐惧)、HAP (开心)、SAD (悲伤)、NEU (中性)；
   - 性别：男 / 女；
   - 年龄：青年 (<35 岁)、中年 (35~55 岁)、老年 (>55 岁)；
4. 预处理文件：preprocess_cremad.py 解析文件名与人口统计信息，生成结构化 CSV 索引

### 3.3 项目目录结构
```
SoundWork/
├── app.py                      # Flask 入口
├── templates/                  # 前端页面
│   └── index.html
│
├── static/                     # 前端资源
│   ├── css/
│   └── js/
│
│  # ------------后端------------
│
├── main.py                    # 核心集成处理
│
├── Data  # 数据集目录
│   ├── CREMA-D
│   └── TIMIT
│
├── models                   # 模型目录
│   ├── __init__.py
│   ├── wav2vec2.py          # wav2vec2模型代码
│   ├── ecapa_tdnn.py        # ECAPA-TDNN模型代码
│   └── whisper_asr.py       # whisper ASR模型代码
│
├── preprocessing            # 数据预处理目录
│   ├── __init__.py
│   ├── compute_emotion_bias.py # 计算情感偏置
│   ├── extract_fbank.py     # 提取FBank特征
│   ├── preprocess_cremad.py # 预处理CREMA-D数据集
│   ├── preprocess_timit.py  # 预处理TIMIT数据集
│   ├── timit_dataset.py     # TIMIT数据集
│   └── wav2vec2_dataset.py  # wav2vec2数据集
│
├── recognition              # 识别/推理模块
│   ├── __init__.py
│   ├── emotion_compensated_reco.py # 情感补偿识别模块
│   ├── speaker_reco.py      # 说话人识别模块
│   └── wav2vec2_reco.py      # 情感识别模块
│
├── emotion_checkpoints      # 情感识别检查点目录
│
├── speaker_checkpoints      # 说话人识别检查点目录
│   ├── speaker_db           # 已注册的说话人数据库
│   ├── best_model.pth       # 最佳模型检查点
│   ├── training_curves.png  # 训练曲线图片
│   └── training_history.json  # 训练历史记录
│
├── tests                    # 测试目录
│   ├── test_ecapa_open.py   # 测试说话人识别模块
│   ├── test_ecapa_tdnn.py   # 测试ECAPA-TDNN模型的关闭集
│   └── test_emotion_compensation.py # 测试情感补偿识别模块
│   
├── training                 # 训练目录
│   ├── train_speaker.py     # 训练说话人识别模型
│   └── train_wav2vec2.py    # 训练情感识别模型
│
├── utils                    # 工具目录
│   ├── __init__.py
│   └── plot_curves.py       # 绘制训练曲线图片
│ 
├── download_model.py        # wav2vec2模型下载脚本
├── .gitignore
├── config.py
└── README.md
```

## 四、核心模块原理与代码解析
按数据预处理 → 三大核心模型 → 补偿算法 → 业务逻辑 → 前后端交互顺序，逐模块讲解原理与代码实现
### 4.1 音频特征提取模块（extract_fbank.py）
#### 4.1.1 原理
- 说话人识别选用FBank（梅尔滤波组特征），相比MFCC保留更多声学细节，更适配深度学习模型。
- 处理流程：音频波形 → 预加重 → 短时傅里叶变换 → 梅尔滤波 → 对数变换 → 长度对齐
#### 4.1.2 关键参数选择
- 采样率 sr=16000Hz：匹配 TIMIT 原始音频标准；
- n_mels=40：TIMIT 数据量小、环境干净，40 维滤波器足以表征特征，80 维会出现高频零值，浪费计算；
- n_fft=512：分帧长度约 32ms，符合语音处理标准（10~40ms）；
- hop_length=256：帧移 16ms；
- max_len=500：统一特征帧长，过长截断、过短补零，保证模型输入维度一致；
- 预加重系数 0.97：补偿语音高频能量衰减
#### 4.1.3 代码核心解析
```python
def extract_fbank(file_path,n_mels=80,max_len=250,sr=16000):
    # 1. 加载音频波形
    y,sr = librosa.load(file_path, sr=sr)
    # 2. 预加重
    y=librosa.effects.preemphasis(y,coef=0.97)
    # 3. 提取梅尔谱
    mel_spec=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=n_mels,n_fft=512,hop_length=256)
    # 4. 对数FBank
    log_mel=librosa.power_to_db(mel_spec,ref=np.max)
    # 5. 长度对齐
    if log_mel.shape[1]<max_len:
        pad_len=max_len-log_mel.shape[1]
        log_mel=np.pad(log_mel,((0,0),(0,pad_len)),mode='constant')
    else:
        log_mel=log_mel[:, :max_len]
    return log_mel
```

### 4.2 说话人识别模块（ECAPA-TDNN）
#### 4.2.1 模型原理（ecapa_tdnn.py）
ECAPA-TDNN 是目前工业界主流声纹识别模型，基于TDNN（时延神经网络）+ Res2Net + SE 通道注意力 + 注意力统计池化架构，专门解决变长语音的声纹提取问题

模型整体结构：
1. 初始卷积层：一维卷积将 FBank 特征升维至 512 维，搭配 BN+ReLU 激活；
2. 三层 SE-Res2Block：核心特征提取单元，融合 Res2Net 多尺度分支与 SE 通道注意力，增强特征表征能力；
3. 维度变换卷积：映射至 1536 维特征；
4. 注意力统计池化：计算加权均值 (μ) 与标准差 (σ)，拼接得到全局语音表征；
5. 全连接层：生成 512 维归一化声纹 Embedding；
6. 分类头：训练阶段用于说话人分类，推理阶段舍弃，仅输出 Embedding

（1）SE_Res2Block 模块
- Res2Net：将特征按通道切分为多分支（scale=8），分支间串行卷积，捕捉多尺度语音特征；
- SEBlock：通道注意力机制，自适应强化有效声学通道、抑制无效噪声通道；
- 残差连接：缓解深度网络梯度消失问题。

（2）注意力统计池化
- 传统池化仅使用均值，本模型结合加权均值 + 加权标准差，全面表征语音全局特征：
$$
\mu = \frac{\sum(x\cdot attn)}{\sum(attn)},\quad \sigma = \sqrt{\frac{\sum(x^2\cdot attn)}{\sum(attn)} - \mu^2}
$$
- 拼接μ与σ作为池化输出
#### 4.2.2 模型前向逻辑（区分训练/推理）
```python
def forward(self,x,is_train=True):
    # 基础卷积+激活
    x=self.conv1(x);x=self.bn1(x);x=self.relu(x)
    # 三层核心残差块
    x=self.layer1(x);x=self.layer2(x);x=self.layer3(x)
    x=self.conv2(x)
    # 注意力统计池化
    attn=self.attention(x)
    mu=torch.sum(x*attn,dim=2)/torch.sum(attn,dim=2)
    sg=torch.sqrt((torch.sum((x**2)*attn,dim=2)/torch.sum(attn,dim=2))-mu**2)
    x=torch.cat((mu,sg),dim=1)
    # 生成归一化Embedding
    x=self.bn2(x);x=self.fc(x)
    embedding=F.normalize(x,p=2,dim=1)
    if not is_train:
        return embedding  # 推理：返回声纹特征
    logits=self.classifier(embedding)
    return logits       # 训练：返回分类概率
```
#### 4.2.3 说话人识别推理逻辑（speaker_reco.py）
该模块封装模型加载、声纹提取、说话人注册、相似度匹配、临时说话人管理全流程
1. 模型加载：加载训练权重时过滤classifier分类层，仅保留特征提取部分
2. 声纹注册：多条语音提取 Embedding 后取均值，存入本地.npy文件构建声纹库
3. 相似度计算：使用余弦相似度衡量两个声纹向量的匹配度
4. 阈值判定：相似度 > 预设阈值 → 匹配到已注册说话人，否则判定为陌生人
5. 临时说话人机制：
   - 首次出现陌生人 → 自动生成Speaker_XX临时 ID，保存声纹
   - 同临时说话人再次出现 → 采用移动平均更新声纹，提升特征稳定性
   - 支持临时说话人转正（样本数≥3、总时长≥5 秒方可转正）

### 4.3 情感/性别/年龄多任务识别模块（Wav2Vec 2.0）
#### 4.3.1 模型原理（wav2vec2_model.py）
Wav2Vec 2.0 是 Facebook 提出的自监督语音预训练大模型，基于海量无标注语音预训练，擅长提取上下文语音特征。用于本项目做多任务微调：
1. 输入：原始 1D 语音波形（16kHz），无需手动提取特征
2. 主干网络：加载开源预训练wav2vec2-base，冻结底层 CNN 特征提取器，节省显存、加速收敛
3. 多任务输出头：设计三个独立全连接分支，分别对应6 类情感、2 类性别、3 类年龄分类任务
4. 池化方式：全局平均池化 (GMP) 将变长时间帧压缩为固定维度句向量
#### 4.3.2 数据集与推理接口
1. 数据集类wav2vec2_dataset.py：读取 CREMA-D 音频，统一重采样至 16kHz、单声道、3 秒长度对齐；
2. 推理类wav2vec2_reco.py：封装音频预处理、模型推理、标签映射，输出中文标签（愤怒 / 快乐 / 男 / 青年等）

### 4.4 语音转写模块（Whisper ASR）
#### 4.4.1 原理（whisper_asr.py）
- 调用 OpenAI Whisper 预训练模型，输入音频直接输出带起止时间戳的文本片段。本项目选用base轻量版本，兼顾速度与准确率
- 核心作用：将连续语音切分为独立语义片段，为后续说话人、情感识别提供时间边界
#### 4.4.2 核心逻辑（whisper_asr.py）
```python
class WhisperASR:
    def __init__(self,model_size="base"):
        self.model=whisper.load_model(model_size)
    def transcribe(self,audio_path):
        # 输出每个片段的start/end/text
        result=self.model.transcribe(audio_path,fp16=False)
        return result["segments"]
```

### 4.5 情感偏移补偿模块
#### 4.5.1 问题背景
同一说话人在不同情绪下，语音音色、频谱会发生偏移，导致声纹 Embedding 变化，降低说话人识别准确率。为此，借鉴语音增强中的谱减法思想，将情感视为"噪声"，设计情感偏移补偿算法，在特征域减去情感偏移
#### 4.5.2 偏移量计算（compute_emotion_bias.py）
1. 以中性情感作为基准，遍历 CREMA-D 数据集；
2. 对每个说话人，计算各类情感 Embedding 与中性 Embedding 的差值（偏移量）；
3. 统计所有样本的平均偏移量、标准差，保存为emotion_bias.pth文件
#### 4.5.3 补偿逻辑（emotion_compensated_reco.py）
继承基础说话人识别类，重写特征提取逻辑：
1. 先通过 Wav2Vec2 识别当前音频情感
2. 读取对应情感的平均偏移量
3. 按补偿强度校正声纹向量：emb_new=emb_raw - strength * emb_bias
   - strength为补偿强度（实验最优值 0.7）
   - 由于补偿强度适中，未产生过减，增益补偿无效，所以没有采用
4. 校正后重新归一化向量，再执行相似度匹配

### 4.6 核心集成模块（main.py）
MeetingDiary类是整个系统的调度中枢，串联所有算法模块：
1. 初始化：加载说话人识别器（带 / 不带补偿）、情感识别器、Whisper 转写模型
2. 音频分段截取：根据 Whisper 时间戳，截取单段语音
3. 串行执行：转写 → 说话人识别 → 情感 / 性别 / 年龄识别 → 情感补偿
4. 后处理：合并相邻同说话人片段，输出结构化会议结果

### 4.7 后端接口与前端交互
#### 4.7.1 Flask 后端（app.py）
封装所有 HTTP 接口，包含：页面路由、音频解析、说话人注册 / 编辑 / 删除、临时说话人转正、缓存清空等接口，完成音频格式转换、跨模块调用、数据返回。
#### 4.7.2 前端（index.html + main.js + style.css）
1. 页面布局：三栏布局（说话人管理区、音频解析控制区、对话展示区）；
2. 核心功能：
   - 文件上传解析、实时麦克风录音（MediaRecorder + VAD 音量检测）；
   - 说话人列表渲染、右键菜单、双击编辑；
   - 对话文本 / 说话人在线编辑、会议日记 TXT 导出；
3. 关键技术点：
   - VAD 语音活动检测：100ms 检测一次音量，静音超过 600ms 判定片段结束；
   - 队列 + 序号机制：解决异步上传解析导致的对话顺序混乱问题；
   - 音频格式兼容：自动转换 WebM 录音文件为标准 WAV


## 五、模型训练过程与参数设置
### 5.1 ECAPA-TDNN 说话人模型训练（train_speaker.py）
#### 5.1.1 训练配置
- 数据集：TIMIT 划分 90% 训练集、10% 验证集；
- 优化器：Adam，学习率lr=0.001；
- 损失函数：交叉熵损失（分类任务）；
- 训练轮数：50 轮；批次大小：32；
- 正则化：梯度裁剪（max_norm=1.0），防止梯度爆炸；
- 保存策略：保留验证集准确率最高的模型为best_model.pth。
#### 5.1.2 训练流程
1. 固定随机种子，保证实验可复现；
2. 加载数据集与模型，部署至 GPU/CPU；
3. 迭代训练：前向传播 → 计算损失 → 反向传播 + 参数更新；
4. 每轮结束执行验证，记录损失与准确率；
5. 保存最优模型与训练历史日志。

### 5.2 Wav2Vec2 多任务模型训练（train_wav2vec2.py）
#### 5.2.1 训练配置
- 数据集：CREMA-D 8:2 划分训练 / 验证集；
- 预训练基座：wav2vec2-base，冻结底层 CNN；
- 学习率：5*10-5；
- 批次大小：16；训练轮数：20 轮；
- 多任务损失加权：Loss=1.5 * Loss_emotion + 0.2 * Loss_gender + 1.0 * Loss_age

### 5.3 情感偏移量计算
运行compute_emotion_bias.py，基于训练好的 ECAPA-TDNN模型提取全量 CREMA-D 声纹，统计各类情感相对中性的偏移向量，输出emotion_bias.pth


## 六、系统功能测试与实验结果分析
### 6.1 ECAPA-TDNN 阈值优选测试（test_ecapa_open.py）
基于 TIMIT 开放集测试，划分50 名注册说话人、118 名未注册说话人，遍历不同相似度阈值，评估两大核心指标：
- 已注册人识别率：正确匹配注册库的比例；
- 未注册人拒绝率：正确判定为陌生人的比例
#### 6.1.1 测试结果
| 阈值 | 已注册识别率 | 未注册拒绝率 | 评价 |
| --- | --- | --- | --- |
| 0.50 | 82.0% | 96.6% | 优秀 |
| 0.52 | 90.0% | 94.1% | 优秀 |
| 0.55 | 72.0% | 98.3% | 良好 |
| 0.60 | 90.0% | 86.4% | 优秀 |
| 0.65 | 52.0% | 99.2% | 较差 |
#### 6.1.2 结论
综合平衡识别率与拒绝率，系统最优阈值设置为 0.52：
- 已注册识别率：90.0%（45/50）
- 未注册拒绝率：94.1%（111/118）

该阈值为系统最终上线使用参数

### 6.2 情感补偿强度测试
遍历补偿强度[0, 1.5]，对比无补偿/有补偿场景下跨情感说话人识别准确率
#### 6.2.1 测试结果
| 补偿强度 | 有补偿准确率 | 提升幅度 |
| --- | --- | --- |
| 0.0 | 20.5% | +0.0% |
| 0.3 | 23.0% | +2.5% |
| 0.5 | 24.5% | +4.0% |
| 0.7 | 26.5% | +6.0% |
| 0.9 | 25.0% | +4.5% |
| 1.5 | 26.0% | +5.5% |
#### 6.2.2 结论
- 最优补偿强度为0.7，识别准确率提升 6.0%，补偿效果显著
- 强度过大（>0.7）会过度校正特征，准确率小幅回落
- 证明情感偏移补偿可有效削弱情绪对声纹特征的干扰

### 6.3 端到端系统功能测试
#### 6.3.1 本地音频解析
输入多人会议 WAV 音频，系统正常输出：时间戳-说话人-情绪-性别-年龄-文本，相邻同说话人片段自动合并，结果与人工判定一致
#### 6.3.2 实时录音测试
- VAD 分段、异步队列工作正常，录音、解析、展示互不阻塞，对话顺序无错乱
- 临时说话人自动注册，多次发言后声纹逐步优化
#### 6.3.3 辅助功能测试
- 说话人注册 / 编辑 / 删除 / 转正：功能正常；
- 对话内容、说话人名称在线编辑：实时生效；
- 会议日记导出：正常生成 TXT 结构化文档

#### 6.4 说话人分割优化
项目初期尝试使用 Wespeaker 做说话人分割，后改为Whisper 时间戳分割，对比如下：
| 对比项 | 旧方案（Wespeaker 分割） | 新方案（Whisper时间戳分割） |
| --- | --- | --- |
| 分割边界 | 固定滑动窗口，边界不准 | 语义句子边界，精准 |
| 识别相似度 | 普遍低于 0.5 | 普遍高于 0.7 |
| 代码复杂度 | 高（大量调参） |	低（直接复用 ASR 结果） |
| 依赖 | Whisper+Wespeaker + ECAPA | Whisper+ ECAPA |
| 依赖评价 | 多模型串联，开销大 | 精简依赖，一模型二用，效率更高 |

优化结论：使用 Whisper 时间戳替代专用分割模型，在精度、效率、可维护性上全面提升


## 七、问题排查、难点与解决方案
结合开发与测试过程中遇到的问题，分类总结如下：
### 7.1 音频分段与顺序问题
- 问题：实时录音异步上传解析，网络延迟导致对话展示顺序错乱
- 解决方案：引入片段序号 + 解析队列 + 等待列表，强制按录音顺序展示结果，异步处理不影响界面顺序
### 7.2 VAD 截断问题
- 问题：静音判断上传片段时，用户立即继续说话，导致语音开头截断
- 解决方案：VAD 检测到静音 > 600ms 后异步上传当前片段，同时立即开启下一段录音，两段流程并行执行。
### 7.3 情绪干扰声纹特征
- 问题：愤怒、开心等情绪下，同一说话人声纹相似度大幅下降
- 解决方案：设计情感偏移补偿算法，实验验证准确率提升 6%
### 7.4 临时说话人管理问题
- 问题：陌生人每次发言都被判定为新用户，无法关联历史声纹
- 解决方案：
  - 设计临时说话人缓存 + 动态权重移动平均更新声纹，样本越多特征越稳定
  - 支持手动转正永久注册
### 7.5 音频格式兼容问题
- 问题：前端 MediaRecorder 输出 WebM 格式，后端无法直接解析
- 解决方案：后端增加格式判断，使用 pydub 自动将 WebM 转标准 WAV


## 八、系统总结
### 8.1 项目完成情况
本项目完整实现多人会议语音智能解析系统，所有预定功能全部落地：
- 完成 ECAPA-TDNN 说话人模型训练、调优，开放集综合指标优秀
- 完成 Wav2Vec2 多任务模型，实现情感、性别、年龄识别
- 提出并实现情感偏移补偿算法，有效提升复杂情绪场景识别精度
- 集成 Whisper 实现高精度语音转写
- 搭建前后端交互系统，支持本地解析、实时录音、说话人管理、日记导出等全功能
- 完成多组对照实验，确定最优阈值、最优补偿强度等关键参数

### 8.2 核心创新点
- 多模型融合架构：将声纹识别、语音分类、语音转写三大语音任务深度集成，面向会议场景落地
- 情感偏移补偿：针对性解决情绪对声纹特征的干扰，属于模型应用层优化
- 临时说话人动态管理：结合移动平均算法，实现陌生人自动识别与特征迭代
- VAD + 队列异步实时解析：兼顾实时性、流畅性与展示顺序，优化用户体验

### 8.3 不足与未来拓展方向
1. 现有不足
  - 仅支持单语种英语，未充分发挥 Whisper 多语言能力
  - 情感补偿强度为固定值，未实现自适应动态调整
  - 未支持批量音频文件批处理
  - 短时语音识别鲁棒性仍有提升空间
2. 未来拓展
  - 增加中文数据集训练，实现中英双语解析
  - 设计自适应补偿强度算法，根据情绪强度动态调整
  - 增加批量解析、历史会议记录存储功能
  - 提高情绪鲁棒性
  - 优化前端 WebSocket 流式传输，进一步降低实时延迟


## 九、参考文献
- [1] Desplanques B, Thienpondt J, Demuynck K. ECAPA-TDNN: Emphasized Channel Attention, Propagation and Aggregation in TDNN Based Speaker Verification [C]. Interspeech, 2020.
- [2] Baevski A, Zhou Y, Mohamed A, et al. Wav2Vec 2.0: A Framework for Self-Supervised Learning of Speech Representations [C]. NeurIPS, 2020.
- [3] Radford A, Kim J W, Xu T, et al. Robust Speech Recognition via Large-Scale Weak Supervision [C]. ICASSP, 2023.
- [4] 韩纪庆，张磊，郑铁然。语音信号处理 [M]. 清华大学出版社.
- [5] CREMA-D: Crowd-Sourced Emotional Multimodal Actors Dataset.
- [6] TIMIT Acoustic-Phonetic Continuous Speech Corpus.




# 缺失
- 数据预处理